# 2026-05-27 `step.san` 生成结果差距分析

本 notebook 以 `data/datasets/components/01_simple/step/san/step.san` 作为完美参考，分析 `data/experiments/step.san` 当前生成结果的不足。

目标不是判断当前生成结果能不能运行，而是评估它距离项目中理想的 San 组件写法还差哪些关键点。


## 1. 参考文件与生成文件

- 参考文件：`/Users/baidu-yangrunsheng/Desktop/CardMigratorSystem/data/datasets/components/01_simple/step/san/step.san`
- 生成文件：`/Users/baidu-yangrunsheng/Desktop/CardMigratorSystem/data/experiments/step.san`


In [ ]:
# 参考 san 文件
<template>
  <div class="step-card" on-click="addSteps">
    <div class="steps-display">
      <span class="steps-value">{{ steps }}</span>
      <span class="steps-unit">步</span>
    </div>
    <div class="steps-label">今日步数</div>
    <div class="hint">点击卡片 +1000 步</div>
  </div>
</template>

<script>
const san = require('san');
const DataTypes = san.DataTypes;

module.exports = san.defineComponent({
  name: 'StepCard',

  dataTypes: {
    initialSteps: DataTypes.number
  },

  initData() {
    return {
      initialSteps: 0,
      steps: 0
    };
  },

  inited() {
    this.data.set('steps', this.data.get('initialSteps'));
  },

  addSteps() {
    this.data.set('steps', this.data.get('steps') + 1000);
  }
});
</script>

<style>
.step-card {
  width: 260px;
  padding: 24px 20px;
  background: linear-gradient(135deg, #43c6ac 0%, #191654 100%);
  border-radius: 20px;
  text-align: center;
  color: white;
  cursor: pointer;
  transition: transform 0.2s;
  font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
}
.step-card:hover {
  transform: translateY(-3px);
  box-shadow: 0 8px 20px rgba(0, 0, 0, 0.2);
}
.steps-display {
  margin-bottom: 12px;
}
.steps-value {
  font-size: 48px;
  font-weight: bold;
}
.steps-unit {
  font-size: 20px;
  margin-left: 4px;
  opacity: 0.8;
}
.steps-label {
  font-size: 18px;
  margin-bottom: 8px;
  opacity: 0.9;
}
.hint {
  font-size: 12px;
  opacity: 0.6;
}
</style>


In [ ]:
# 当前生成出来的 step.san 源代码
san.defineComponent({
    template: `<div class="step-card" on-click="addSteps">
        <div class="steps-display">
            <span class="steps-value">{{steps}}</span>
            <span class="steps-unit">步</span>
        </div>
        <div class="steps-label">今日步数</div>
        <div class="hint">点击卡片 +1000 步</div>
    </div>`,
    
    initData() {
        return {
            steps: this.data.get('initialSteps') || 0
        };
    },
    
    dataTypes: {
        initialSteps: 'number'
    },
    
    methods: {
        addSteps() {
            const current = this.data.get('steps');
            this.data.set('steps', current + 1000);
        }
    }
});


## 2. 总体结论

当前生成结果已经抓住了这个组件最核心的迁移语义：

- 根节点结构正确
- 点击事件正确迁移为 `on-click`
- `steps` 的读写逻辑基本正确
- 模板文本和类名基本保留

但是如果以参考文件为“完美版本”，当前 `data/experiments/step.san` 仍然有几个明显不足，会影响代码风格一致性、运行稳健性和样式完整性。


## 3. 不足一：缺少完整的 SFC 结构

参考文件是标准的 San 单文件组件结构：

- `<template>`
- `<script>`
- `<style>`

而当前生成结果只有一段 `san.defineComponent({...})`，缺少：

- `<template>` 包裹
- `<script>` 包裹
- `<style>` 样式块

这意味着当前结果更像一段“组件实现片段”，而不是项目中可直接落地替换的完整 `.san` 文件。


## 4. 不足二：缺少 `require('san')` 与 `module.exports`

参考文件在脚本部分明确写了：

- `const san = require('san');`
- `const DataTypes = san.DataTypes;`
- `module.exports = san.defineComponent({...})`

而当前生成结果直接从：

- `san.defineComponent({...})`

开始，没有提供模块导出与依赖引入。

影响：

- 在项目代码库中通常不能直接作为完整组件使用
- 需要人工补 `san` 依赖和导出语句


## 5. 不足三：`dataTypes` 类型写法不符合参考实现

参考文件：

- `initialSteps: DataTypes.number`

当前生成结果：

- `initialSteps: 'number'`

这说明模型虽然理解了 prop 是 number，但没有对齐项目中更标准的 San 类型声明写法。

影响：

- 可读性和项目一致性较差
- 如果项目要求统一使用 `DataTypes`，则仍需人工修正


## 6. 不足四：`initData()` 初始化语义不够稳妥

参考实现采用两段式初始化：

1. `initData()` 先给出稳定默认值
   - `initialSteps: 0`
   - `steps: 0`
2. `inited()` 再执行 prop -> state 的同步
   - `this.data.set('steps', this.data.get('initialSteps'))`

当前生成结果把逻辑写成：

- `steps: this.data.get('initialSteps') || 0`

不足点：

- `initData()` 里直接读取 `this.data.get('initialSteps')`，风格上不如参考实现稳定
- 没有显式保留 `initialSteps` 默认值
- 把 prop 初始化和 state 初始化混在一起，不如 `initData + inited` 分工清晰

这说明目前 `SSM` 虽然知道 `steps` 来源于 `initialSteps`，但还不足以强约束模型生成参考实现那种更规范的生命周期初始化方式。


## 7. 不足五：缺少 `name` 字段

参考文件明确包含：

- `name: 'StepCard'`

当前生成结果没有输出组件名。

影响：

- 组件调试信息与项目一致性下降
- 从 `metadata.component_name` 到最终代码的映射没有被完整保留


## 8. 不足六：样式完全缺失

参考文件保留了完整样式：

- `.step-card`
- `.step-card:hover`
- `.steps-display`
- `.steps-value`
- `.steps-unit`
- `.steps-label`
- `.hint`

而当前生成结果虽然保留了类名，但完全没有输出 `<style>`。

这是当前结果和参考文件之间最大的差距之一。

影响：

- 页面视觉效果无法还原
- `style_model` / `styles` 中已有的大量信息没有真正落实到生成结果里


## 9. 不足七：方法组织形式与项目参考风格不一致

参考文件直接把方法定义在组件对象顶层：

- `addSteps() { ... }`

而当前生成结果采用：

- `methods: { addSteps() { ... } }`

这更像 Vue Options API 的写法迁移残留，而不是完全贴近参考 San 组件风格。

对运行未必一定有问题，但从“项目中的理想 San 写法”来看，不够干净。


## 10. 不足八：模板细节风格尚未完全对齐

模板结构总体正确，但仍有细节差异：

- 参考文件插值写法是 `{{ steps }}`
- 当前生成结果是 `{{steps}}`

这类问题不影响功能，但说明生成结果还没有完全对齐项目中更规范的代码风格。


## 11. 根因判断

从这次对比可以看出，当前 `SSM` 对于“行为语义”已经足够：

- 事件
- 数据绑定
- props
- 方法读写

但是对“最终代码组织形式”的约束还不够强，主要体现在：

- 没强约束必须输出完整 SFC 三段结构
- 没强约束必须引入 `san`、导出 `module.exports`
- 没强约束 `dataTypes` 使用 `DataTypes.number`
- 没强约束 `styles` 必须落实为 `<style>` 输出
- 没强约束 `initialSteps -> steps` 应通过 `inited()` 同步


## 12. 结论

如果以参考文件为完美标准，当前 `data/experiments/step.san` 的主要不足可以概括为：

1. 不是完整的 `.san` 单文件组件结构
2. 缺少 `san` 引入和模块导出
3. `dataTypes` 类型声明不规范
4. `initData()` 与 `inited()` 的初始化分工缺失
5. 缺少组件名 `name`
6. 缺少完整样式输出
7. 方法组织形式残留 Vue 风格
8. 模板格式化细节未对齐参考风格

但同时也要看到：它已经生成出了正确的核心结构和状态更新逻辑，这说明当前 `SSM` 已经能支撑“基础可用”的 San 代码生成，只是距离“项目级高质量对齐输出”还有一段距离。
